In [2]:
!pip install -U bitsandbytes transformers peft accelerate trl datasets pypdf pandas

import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Setup Environment
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

# 1. CONFIGURATION
csv_file = "/content/Dataset.csv"
cv_col = "Context"
roast_col = "Response"
model_id = "unsloth/llama-3-8b-bnb-4bit"
output_dir = "roaster_v1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 2. TOKENIZER & MODEL LOADING
print("⏳ Loading Model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# 3. PREPARE FOR QLoRA
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)

# 4. DATASET PREPARATION (With Validation Split)
print("📊 Preparing Dataset...")
df = pd.read_csv(csv_file)
dataset = Dataset.from_pandas(df)
split_dataset = dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

def format_prompt(example):
    return (
        "### CV Context:\n"
        + str(example[cv_col])
        + "\n\n### Roast:\n"
        + str(example[roast_col])
    )

# 5. TRAINING CONFIGURATION
sft_config = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="steps",
    eval_steps=10,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
)

# 6. TRAIN
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=format_prompt,
)

print("🚀 Starting Training...")
trainer.train()

# 7. SAVE
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Training Finished & Saved to {output_dir}!")

⏳ Loading Model and Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:259: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

📊 Preparing Dataset...


Applying formatting function to train dataset:   0%|          | 0/52 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

🚀 Starting Training...


Step,Training Loss,Validation Loss
10,2.563415,2.034569
20,1.774187,1.910138
30,1.478421,1.846769


✅ Training Finished & Saved to roaster_v1!


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 1. Configuration
base_model_id = "unsloth/llama-3-8b-bnb-4bit"
lora_model_path = "/content/roaster_v1"

# 2. Setup Quantization & Load Base Model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("⏳ Loading base model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

# 3. Load Fine-Tuned Adapters
print("🔧 Applying your 'Roaster' LoRA weights...")
model = PeftModel.from_pretrained(base_model, lora_model_path)
model.eval()

# 4. The Test Data
# You can change this text to anything you want to test!
test_cv_context = "AI/ML Engineer. Built a custom RAG system using LangChain and deployed a YouTube reply bot using Python. Strong skills in API integration and prompt engineering."

# The prompt MUST match your training format exactly
prompt = f"### CV Context:\n{test_cv_context}\n\n### Roast:\n"

# 5. Generate the Output
print("🔥 Generating roast...\n")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,        # 0.7 gives a good mix of creativity and staying on-topic
        top_p=0.9,
        repetition_penalty=1.1, # Prevents the model from repeating the same insult
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only the newly generated tokens
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("="*50)
print("TEST CV CONTEXT:")
print(test_cv_context)
print("-" * 50)
print("MODEL ROAST:")
print(response.strip())
print("="*50)

⏳ Loading base model and tokenizer...


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:259: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

🔧 Applying your 'Roaster' LoRA weights...


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔥 Generating roast...

TEST CV CONTEXT:
AI/ML Engineer. Built a custom RAG system using LangChain and deployed a YouTube reply bot using Python. Strong skills in API integration and prompt engineering.
--------------------------------------------------
MODEL ROAST:
Your model answers questions perfectly—until it realizes the context is real life and becomes sentient. Now it only responds with existential dread.


In [4]:
# 1. Compress the model folder into a zip file
!zip -r roaster_v1.zip roaster_v1/

# 2. Trigger the download to your local machine
from google.colab import files
print("⏳ Preparing download... ")
files.download('roaster_v1.zip')

  adding: roaster_v1/ (stored 0%)
  adding: roaster_v1/tokenizer_config.json (deflated 45%)
  adding: roaster_v1/checkpoint-39/ (stored 0%)
  adding: roaster_v1/checkpoint-39/tokenizer_config.json (deflated 43%)
  adding: roaster_v1/checkpoint-39/trainer_state.json (deflated 71%)
  adding: roaster_v1/checkpoint-39/training_args.bin (deflated 53%)
  adding: roaster_v1/checkpoint-39/adapter_model.safetensors (deflated 22%)
  adding: roaster_v1/checkpoint-39/rng_state.pth (deflated 26%)
  adding: roaster_v1/checkpoint-39/optimizer.pt (deflated 11%)
  adding: roaster_v1/checkpoint-39/scheduler.pt (deflated 62%)
  adding: roaster_v1/checkpoint-39/adapter_config.json (deflated 57%)
  adding: roaster_v1/checkpoint-39/README.md (deflated 65%)
  adding: roaster_v1/checkpoint-39/tokenizer.json (deflated 85%)
  adding: roaster_v1/checkpoint-13/ (stored 0%)
  adding: roaster_v1/checkpoint-13/tokenizer_config.json (deflated 43%)
  adding: roaster_v1/checkpoint-13/trainer_state.json (deflated 62%)
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>